In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (root_mean_squared_error, mean_absolute_error, r2_score)


In [2]:

# Load the dataset
df = pd.read_csv(r"D:\\MLops Day 1\\Data\\data.csv")
df

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9
...,...,...,...,...
195,38.2,3.7,13.8,7.6
196,94.2,4.9,8.1,14.0
197,177.0,9.3,6.4,14.8
198,283.6,42.0,66.2,25.5


In [3]:
x = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=56)

In [4]:
#first we config Mlflow to store all expermental tracking metadata inside a local sqlite database name mlflow.db located in the current working directory


In [5]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

#### **then we crete an experiment name advertising sales prediction**


In [6]:
mlflow.set_experiment("Advertising_Sales_Prediction")

<Experiment: artifact_location='file:d:/MLops Day 1/Notebooks/mlruns/1', creation_time=1788339981480, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788339981480, lifecycle_stage='active', name='Advertising_Sales_Prediction', tags={}, trace_location=None, workspace='default'>

#### THEN WE CREATE A RUN INSIDE THE EXPERIMENT ADVERTISING SALES PRIDICTION 

In [7]:
with mlflow.start_run(run_name="Linear Regression"):
    model = LinearRegression()
    model.fit(xtrain,ytrain)
    y_pred = model.predict(xtest)
    rmse = root_mean_squared_error(ytest, y_pred)
    r2 = r2_score(ytest, y_pred)

    # parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size",0.2)
    mlflow.log_param("random_state", 56)
    
    #metrix
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

##### NOW WE CREATE  ANOTHER RUN USING RIDGE REGRESSION MODEL

In [8]:
with mlflow.start_run(run_name  = "Ridge Regression"):
    model = Ridge(alpha=1.0)
    model.fit(xtrain,ytrain)
     
    ypred = model.predict(xtest)
    rmse = root_mean_squared_error(ytest,y_pred)
    r2 = r2_score(ytest,y_pred)

     #log the parameters
    mlflow.log_param("model_type", "Ridge Regression")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size",0.2)
    mlflow.log_param("random_state",56)

     #log matrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2",r2)

     # log Artifacts
    mlflow.sklearn.log_model(sk_model = model, name = "ridge_reg_model")


#### ARTIFACTS ARE THE CONCRETE OUTPUT FILES GENERATED BY A RUN

###### NOW WE USE THE AURTOLOGGING FEATURE IN MLFLOW 

###### AUTOLOGGING ALLOWS MLFLOW TO AUTOMATICALLY CAPTURE MUCH OF THE INFORMATION 

In [9]:
# TUEN THE SCIKITLEARN AUTOLOGGING
mlflow.sklearn.autolog()

In [10]:
with mlflow.start_run(run_name="Random Forest Autolog") as run:
    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    model.fit(xtrain,ytrain)
    test_pred = model.predict(xtest)
    test_rmse = root_mean_squared_error(ytest,test_pred)
    test_mae = mean_absolute_error(ytest,test_pred)
    test_r2 = r2_score(ytest,test_pred)
    
    #customer project metrics
    mlflow.log_metrics({
        "test_rmse" : test_rmse,
        "test_mae" : test_mae,
        "test_r2" : test_r2
    })

2026/09/04 20:50:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


#### REGISTERING THE MODEL

- THE MLFLOW MODEL TEGISTRY IS A CENTRAL HUB (LIKE AN APP STORE OR AATALOG) TO KEEP ALL PRODUCTION READY MODELS IN ONE SHARED.
- AMONG ALL THE EXPRIMENTS YOU PERFORM REGISTOR THE INFAL SELECTION MODEL .
- IN  A PRODUCTION ENVIROMETS THE MODELS ARE CONTINOUSLY TRAINED . THIS MEANS THE REGISTERED MODEL WORLD HAVE MANY VERSIONS 
- ADVERTISING_SALES_MODEL
-VERSION 1 --> VERSION 2 ---> VERSION 3 ---> VERSION 4

In [11]:
# FOR REGISTERING THE MODEL  WE REQUIRE THE MODEL URI
# URI IS A UNIQUE IDENTIFIER FOR THE MODEL
run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"
print(model_uri)

runs:/2796c2b919da44c4a48d4296689eed76/model


#### NOW JUST REGISTER THE MODEL 


In [12]:
registered_model = mlflow.register_model(
    model_uri = model_uri,
    name = "Advertising_Sales_Model"
)

Registered model 'Advertising_Sales_Model' already exists. Creating a new version of this model...
2026/09/04 20:50:31 WARNING mlflow.tracking._model_registry.fluent: Run with id 2796c2b919da44c4a48d4296689eed76 has no artifacts at artifact path 'model', registering model based on models:/m-2e51e867a64c4d909afe57e18168aa05 instead
Created version '2' of model 'Advertising_Sales_Model'.


#### MODEL ALIASES

- MODEL ALIAISNG GIVE A NICKNAME (LIKE "CURRENT BEST" PR "PRODUCTION') TO A SPECIFIC VERSION OF THE REGISTER MODEL 
- INSTEAD OF TYPING EXACT NUMBERS LIKE VERSION 1 AND VERSION 2 OR VERSION 15 , YOU JUST USE THE NICKNAME 

In [13]:
from mlflow import MlflowClient 

client = MlflowClient()

# ASSIGN AN ALIAS TP A SPECIFIC VERSION 
# SETS THE ALIES "CHAMPION"  TO VERSION 2 OF "FRAUD_DETECTOR"
client.set_registered_model_alias(
    name = "Advertising_Sales_Model",
    alias = "champion",
    version = "1"
)

### LOADING THE REGISTER MODEL 

In [14]:
model = mlflow.sklearn.load_model(
    "models:/Advertising_Sales_Model@champion")
    # Generate new prisictions
new_data = pd.DataFrame({
    "TV" : [150.0],
    "Radio" : [25.0],
    "Newspaper" : [30.0]
})
print(model.predict(new_data))


[14.2199203]
